In [1]:
import lightgbm
import mlflow
import os
from dotenv import load_dotenv
import sys
import numpy as np
import json

In [ ]:
pd.set_option("display.max_columns", 100)
load_dotenv()
data_path = os.getenv("DATA_PATH")
src_path = os.getenv("SRC_PATH")
sys.path.append(src_path)
json_path = os.path.join(data_path, "processed/split_info.json")
with open(json_path) as f:
    json_info = json.load(f)
train_end = json_info.get("train_end")
val_end = json_info.get("validation_end")
from about_data.data_load import load_df
from about_data.split import temporal_split
from features.engineering import create_d_features, create_advanced_time_features, add_distance_features, add_interaction_features, add_all_features
from model.preprocessor_pipe_evalueate import create_pipeline, evaluate_model, get_preprocessor

full_df = load_df(data_path)
train, val, test = temporal_split(full_df, train_end, val_end)

In [ ]:
map_dfs = {"train": train, "val": val, "test": test}

train_base = create_d_features(train)
val_base = create_d_features(val)

train_features_dfs = {
    "d": train_base,
    "time": create_advanced_time_features(train_base),
    "distance": add_distance_features(train_base),
    "interaction": add_interaction_features(train_base),
    "all": add_all_features(train),
}
val_features_dfs = {
    "d": val_base,
    "time": create_advanced_time_features(val_base),
    "distance": add_distance_features(val_base),
    "interaction": add_interaction_features(val_base),
    "all": add_all_features(val),
}

In [4]:
len(train_features_dfs)

5

In [5]:
y_datasets = {}

for name, sample_df in map_dfs.items():
    y_datasets[name] = sample_df['isFraud']

In [6]:
model = lightgbm.LGBMClassifier(n_estimators=300, learning_rate=0.05, objective="binary", metric='average_precision', random_state=42, n_jobs=-1)

In [ ]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("fraud-detection-feature-engineering")

results = []

for name in train_features_dfs:
    with mlflow.start_run(run_name=f"lightgbm_{name}_features"):
        X_train = train_features_dfs[name]
        X_val = val_features_dfs[name]

        pipe = create_pipeline(model, get_preprocessor(X_train))
        pipe.fit(X_train, y_datasets["train"])
        metrics = evaluate_model(pipe, X_val, y_datasets["val"])

        mlflow.log_param("dataset", name)
        mlflow.log_param("feature_count", X_train.shape[1])
        mlflow.log_param("model", "lightgbm")
        mlflow.log_metrics(metrics)

        results.append({"feature_set": name, **metrics})

[LightGBM] [Info] Number of positive: 14538, number of negative: 398840
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.085608 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 10868
[LightGBM] [Info] Number of data points in the train set: 413378, number of used features: 4094
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035169 -> initscore=-3.311794
[LightGBM] [Info] Start training from score -3.311794
🏃 View run lightgbm_d_features at: http://127.0.0.1:5000/#/experiments/4/runs/be96016d487b41c6aa17cc17d94cb478
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
[LightGBM] [Info] Number of positive: 14538, number of negative: 398840
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.092572 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you

C:\Users\bben2\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['dist_diff' 'dist_sum']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


[LightGBM] [Info] Number of positive: 14538, number of negative: 398840
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.076651 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 11152
[LightGBM] [Info] Number of data points in the train set: 413378, number of used features: 4097
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035169 -> initscore=-3.311794
[LightGBM] [Info] Start training from score -3.311794


C:\Users\bben2\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['dist_diff' 'dist_sum']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\bben2\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['dist_diff' 'dist_sum']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


🏃 View run lightgbm_distance_features at: http://127.0.0.1:5000/#/experiments/4/runs/5d279fd33e564e049834545d36e8dd97
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
[LightGBM] [Info] Number of positive: 14538, number of negative: 398840
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.081809 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 13864
[LightGBM] [Info] Number of data points in the train set: 413378, number of used features: 5453
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035169 -> initscore=-3.311794
[LightGBM] [Info] Start training from score -3.311794
🏃 View run lightgbm_interaction_features at: http://127.0.0.1:5000/#/experiments/4/runs/b2f7408cc4a546e8a0397cb46cdd4031
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


C:\Users\bben2\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['dist_diff' 'dist_sum']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


[LightGBM] [Info] Number of positive: 14538, number of negative: 398840
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.127409 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 13594
[LightGBM] [Info] Number of data points in the train set: 413378, number of used features: 5453
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035169 -> initscore=-3.311794
[LightGBM] [Info] Start training from score -3.311794


C:\Users\bben2\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['dist_diff' 'dist_sum']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\bben2\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['dist_diff' 'dist_sum']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


🏃 View run lightgbm_all_features at: http://127.0.0.1:5000/#/experiments/4/runs/87e7413f849a44738dc58e3a1ccf9ee0
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


In [8]:
pd.DataFrame(results)

,pr_auc,roc_auc,precision,recall,f1
0,0.308680,0.869374,0.602041,0.096976,0.167044
1,0.370914,0.876728,0.675150,0.148258,0.243127
2,0.372754,0.878931,0.672673,0.147272,0.241640
3,0.374009,0.880371,0.674491,0.141683,0.234175
4,0.304854,0.868914,0.592105,0.088757,0.154374
